# bronze_dq_engine\nGenerated from 02_processing/bronze_dq_engine.py for Databricks notebook import.\n

In [ ]:
"""Bronze-layer data quality engine.\n\nThis module validates raw/landing payloads before conformance using optional\nmetadata embedded in source_options_json under the `bronze_dq` key.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Any\n\n\ndef bronze_dq_config(source_options: dict[str, Any] | None) -> dict[str, Any]:\n    cfg = (source_options or {}).get("bronze_dq", {})\n    return cfg if isinstance(cfg, dict) else {}\n\n\ndef validate_bronze_dq_config(cfg: dict[str, Any]) -> list[str]:\n    errors: list[str] = []\n\n    list_keys = ["required_columns", "expected_columns", "non_nullable_columns", "logical_checks"]\n    for key in list_keys:\n        if key in cfg and not isinstance(cfg[key], list):\n            errors.append(f"bronze_dq.{key} must be a list")\n\n    dict_keys = ["expected_types", "regex_patterns", "allowed_values", "length_constraints", "range_constraints", "null_thresholds"]\n    for key in dict_keys:\n        if key in cfg and not isinstance(cfg[key], dict):\n            errors.append(f"bronze_dq.{key} must be an object")\n\n    uniqueness = cfg.get("uniqueness_constraints", [])\n    if uniqueness and not isinstance(uniqueness, list):\n        errors.append("bronze_dq.uniqueness_constraints must be a list")\n\n    freshness = cfg.get("freshness", {})\n    if freshness and not isinstance(freshness, dict):\n        errors.append("bronze_dq.freshness must be an object")\n\n    volume = cfg.get("volume", {})\n    if volume and not isinstance(volume, dict):\n        errors.append("bronze_dq.volume must be an object")\n\n    for col, threshold in (cfg.get("null_thresholds", {}) or {}).items():\n        try:\n            v = float(threshold)\n        except (TypeError, ValueError):\n            errors.append(f"bronze_dq.null_thresholds.{col} must be numeric")\n            continue\n        if v < 0 or v > 1:\n            errors.append(f"bronze_dq.null_thresholds.{col} must be between 0 and 1")\n\n    return errors\n\n\ndef dataset_schema_checks(df, cfg: dict[str, Any]) -> list[dict[str, Any]]:\n    results: list[dict[str, Any]] = []\n\n    cols = set(df.columns)\n    required = set(cfg.get("required_columns", []))\n    missing_required = sorted(required - cols)\n    results.append(\n        {\n            "check_name": "required_columns",\n            "status": "PASS" if not missing_required else "FAIL",\n            "details": {"missing_columns": missing_required},\n        }\n    )\n\n    expected = set(cfg.get("expected_columns", []))\n    if expected:\n        extra = sorted(cols - expected)\n        results.append(\n            {\n                "check_name": "extra_columns",\n                "status": "PASS" if not extra else "FAIL",\n                "details": {"extra_columns": extra},\n            }\n        )\n\n    expected_types = cfg.get("expected_types", {}) or {}\n    type_map = {f.name: f.dataType.simpleString() for f in df.schema.fields}\n    type_mismatch = {}\n    for col, expected_type in expected_types.items():\n        actual = type_map.get(col)\n        if actual and str(actual).lower() != str(expected_type).lower():\n            type_mismatch[col] = {"expected": expected_type, "actual": actual}\n    results.append(\n        {\n            "check_name": "data_types",\n            "status": "PASS" if not type_mismatch else "FAIL",\n            "details": {"mismatch": type_mismatch},\n        }\n    )\n\n    return results\n\n\ndef row_level_checks(df, cfg: dict[str, Any]):\n    from pyspark.sql import functions as F\n\n    result = df.withColumn("bronze_dq_status", F.lit("PASS")).withColumn("bronze_dq_failed_check", F.lit(None).cast("string"))\n\n    for col in cfg.get("non_nullable_columns", []):\n        if col in df.columns:\n            failed = F.col(col).isNull()\n            result = result.withColumn("bronze_dq_status", F.when(failed, F.lit("FAIL")).otherwise(F.col("bronze_dq_status"))).withColumn(\n                "bronze_dq_failed_check",\n                F.when(failed & F.col("bronze_dq_failed_check").isNull(), F.lit(f"non_null:{col}")).otherwise(F.col("bronze_dq_failed_check")),\n            )\n\n    for col, values in (cfg.get("allowed_values", {}) or {}).items():\n        if col in df.columns and isinstance(values, list) and values:\n            failed = ~F.col(col).isin([v for v in values])\n            result = result.withColumn("bronze_dq_status", F.when(failed, F.lit("FAIL")).otherwise(F.col("bronze_dq_status"))).withColumn(\n                "bronze_dq_failed_check",\n                F.when(failed & F.col("bronze_dq_failed_check").isNull(), F.lit(f"allowed_values:{col}")).otherwise(F.col("bronze_dq_failed_check")),\n            )\n\n    for col, pattern in (cfg.get("regex_patterns", {}) or {}).items():\n        if col in df.columns and pattern:\n            failed = (~F.col(col).rlike(str(pattern))) & F.col(col).isNotNull()\n            result = result.withColumn("bronze_dq_status", F.when(failed, F.lit("FAIL")).otherwise(F.col("bronze_dq_status"))).withColumn(\n                "bronze_dq_failed_check",\n                F.when(failed & F.col("bronze_dq_failed_check").isNull(), F.lit(f"regex:{col}")).otherwise(F.col("bronze_dq_failed_check")),\n            )\n\n    for col, spec in (cfg.get("length_constraints", {}) or {}).items():\n        if col not in df.columns or not isinstance(spec, dict):\n            continue\n        min_len = spec.get("min")\n        max_len = spec.get("max")\n        failed = F.lit(False)\n        if min_len is not None:\n            failed = failed | (F.length(F.col(col)) < F.lit(int(min_len)))\n        if max_len is not None:\n            failed = failed | (F.length(F.col(col)) > F.lit(int(max_len)))\n        result = result.withColumn("bronze_dq_status", F.when(failed, F.lit("FAIL")).otherwise(F.col("bronze_dq_status"))).withColumn(\n            "bronze_dq_failed_check",\n            F.when(failed & F.col("bronze_dq_failed_check").isNull(), F.lit(f"length:{col}")).otherwise(F.col("bronze_dq_failed_check")),\n        )\n\n    for col, spec in (cfg.get("range_constraints", {}) or {}).items():\n        if col not in df.columns or not isinstance(spec, dict):\n            continue\n        min_val = spec.get("min")\n        max_val = spec.get("max")\n        cast_col = F.col(col).cast("double")\n        failed = F.lit(False)\n        if min_val is not None:\n            failed = failed | (cast_col < F.lit(float(min_val)))\n        if max_val is not None:\n            failed = failed | (cast_col > F.lit(float(max_val)))\n        result = result.withColumn("bronze_dq_status", F.when(failed, F.lit("FAIL")).otherwise(F.col("bronze_dq_status"))).withColumn(\n            "bronze_dq_failed_check",\n            F.when(failed & F.col("bronze_dq_failed_check").isNull(), F.lit(f"range:{col}")).otherwise(F.col("bronze_dq_failed_check")),\n        )\n\n    for expr in cfg.get("logical_checks", []):\n        if not expr:\n            continue\n        failed = ~F.expr(str(expr))\n        result = result.withColumn("bronze_dq_status", F.when(failed, F.lit("FAIL")).otherwise(F.col("bronze_dq_status"))).withColumn(\n            "bronze_dq_failed_check",\n            F.when(failed & F.col("bronze_dq_failed_check").isNull(), F.lit(f"logical:{expr}")).otherwise(F.col("bronze_dq_failed_check")),\n        )\n\n    valid_df = result.filter("bronze_dq_status = 'PASS'")\n    reject_df = result.filter("bronze_dq_status = 'FAIL'")\n    return result, valid_df, reject_df\n\n\ndef dataset_quality_checks(df, cfg: dict[str, Any]) -> list[dict[str, Any]]:\n    from pyspark.sql import functions as F\n\n    results: list[dict[str, Any]] = []\n    total_rows = df.count()\n\n    for col, threshold in (cfg.get("null_thresholds", {}) or {}).items():\n        if col not in df.columns:\n            continue\n        null_rows = df.filter(F.col(col).isNull()).count()\n        ratio = (null_rows / total_rows) if total_rows else 0\n        max_ratio = float(threshold)\n        results.append(\n            {\n                "check_name": f"null_threshold:{col}",\n                "status": "PASS" if ratio <= max_ratio else "FAIL",\n                "details": {"ratio": ratio, "max_ratio": max_ratio},\n            }\n        )\n\n    for entry in (cfg.get("uniqueness_constraints", []) or []):\n        cols = entry if isinstance(entry, list) else [entry]\n        cols = [c for c in cols if c in df.columns]\n        if not cols:\n            continue\n        distinct = df.select(*cols).distinct().count()\n        dupes = max(total_rows - distinct, 0)\n        results.append(\n            {\n                "check_name": f"uniqueness:{','.join(cols)}",\n                "status": "PASS" if dupes == 0 else "FAIL",\n                "details": {"duplicate_rows": dupes},\n            }\n        )\n\n    freshness = cfg.get("freshness", {}) or {}\n    freshness_col = freshness.get("column")\n    max_age_hours = freshness.get("max_age_hours")\n    if freshness_col in df.columns and max_age_hours is not None:\n        latest_ts = df.select(F.max(F.col(freshness_col)).alias("latest_ts")).collect()[0]["latest_ts"]\n        age_hours = None\n        if latest_ts is not None:\n            age_hours = df.select((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.lit(latest_ts))) / 3600.0).collect()[0][0]\n        status = "PASS" if age_hours is not None and age_hours <= float(max_age_hours) else "FAIL"\n        results.append(\n            {\n                "check_name": f"freshness:{freshness_col}",\n                "status": status,\n                "details": {"age_hours": age_hours, "max_age_hours": float(max_age_hours)},\n            }\n        )\n\n    volume = cfg.get("volume", {}) or {}\n    min_rows = volume.get("min_rows")\n    max_rows = volume.get("max_rows")\n    if min_rows is not None or max_rows is not None:\n        status = True\n        if min_rows is not None:\n            status = status and total_rows >= int(min_rows)\n        if max_rows is not None:\n            status = status and total_rows <= int(max_rows)\n        results.append(\n            {\n                "check_name": "volume",\n                "status": "PASS" if status else "FAIL",\n                "details": {"row_count": total_rows, "min_rows": min_rows, "max_rows": max_rows},\n            }\n        )\n\n    return results\n\n\ndef run_bronze_dq_checks(df, source_options: dict[str, Any] | None) -> dict[str, Any]:\n    cfg = bronze_dq_config(source_options)\n    cfg_errors = validate_bronze_dq_config(cfg)\n\n    schema_results = dataset_schema_checks(df, cfg)\n    annotated_df, valid_df, reject_df = row_level_checks(df, cfg)\n    dataset_results = dataset_quality_checks(annotated_df, cfg)\n\n    return {\n        "config_errors": cfg_errors,\n        "results": schema_results + dataset_results,\n        "annotated_df": annotated_df,\n        "valid_df": valid_df,\n        "reject_df": reject_df,\n    }\n